# fase_4 - script_hanif Migration

This notebook handles migration of database from old DB to new DB for fase 4.

**Purpose**: Migrasi data Siswa dan Mitra dengan Mapping Kolom Spesifik

In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import re
from datetime import datetime
import pickle
import json
sys.path.append(os.path.abspath('..'))
from config import get_db_config
import warnings
warnings.filterwarnings('ignore')

## 1. Connect ke Database

In [2]:
config = get_db_config()
db_old = mysql.connector.connect(**config['db_old'])
cursor_old = db_old.cursor(dictionary=True)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)
print(f"Connected to {config['db_old']['database']} and {config['db_new']['database']}")

Connected to dataleap_v5_example and dataleap_v5_migration


## 2. Ambil Data dari DB Lama

In [3]:
hanif_tables_map = [
    ('siswa', 'siswa'),
    ('siswa_keluar', 'siswa_keluar'),
    ('mitra', 'mitra'),
    ('mitra_note', 'mitra_progres'),
    ('mitra_users', 'kemitraan_verifikator'),
    ('siswamitra', 'siswa_mitra'),
    ('siswa_keluar_mitra', 'siswa_mitra_keluar')
]

raw_data = {}
for old_t, new_t in hanif_tables_map:
    try:
        cursor_old.execute(f"SELECT * FROM `{old_t}`")
        raw_data[old_t] = cursor_old.fetchall()
        print(f"✅ {old_t} loaded: {len(raw_data[old_t])} records")
    except Exception as e:
        print(f"❌ ERROR loading {old_t}: {e}")

✅ siswa loaded: 1469 records
✅ siswa_keluar loaded: 556 records
✅ mitra loaded: 22 records
✅ mitra_note loaded: 296 records
✅ mitra_users loaded: 228 records
✅ siswamitra loaded: 0 records
✅ siswa_keluar_mitra loaded: 0 records


## 3. Transform Data (Mapping Berdasarkan hanif_mapping.md)

In [4]:
transformed_dfs = {}

# --- HELPER FUNCTIONS ---
def extract_int(s):
    if pd.isna(s) or not str(s).strip(): return None
    nums = re.findall(r'\d+', str(s))
    return int(nums[0]) if nums else None

def extract_chars(s):
    if pd.isna(s) or not str(s).strip(): return None
    return re.sub(r'\d+', '', str(s)).strip()

def convert_ya_tidak(s):
    if pd.isna(s): return 0
    val = str(s).strip().lower()
    return 1 if val == 'ya' else 0

# --- TRANSFORMATION ---

# 1. siswa -> siswa
if 'siswa' in raw_data:
    df = pd.DataFrame(raw_data['siswa'])
    df['idmitra_int'] = df['idmitra'].apply(extract_int)

    # Fetch region tables from both databases to build hierarchical name mappings (Read-only lookup)
    cursor_old.execute("SELECT idprovinsi, nama FROM provinsi")
    df_old_prov = pd.DataFrame(cursor_old.fetchall())
    cursor_new.execute("SELECT id_provinsi, nama_provinsi FROM provinsi")
    df_new_prov = pd.DataFrame(cursor_new.fetchall())

    cursor_old.execute("SELECT idkabupaten, idprovinsi, name FROM kabupaten")
    df_old_kab = pd.DataFrame(cursor_old.fetchall())
    cursor_new.execute("SELECT id_kabupaten, id_provinsi, nama_kabupaten FROM kabupaten")
    df_new_kab = pd.DataFrame(cursor_new.fetchall())

    cursor_old.execute("SELECT idkecamatan, idkabupaten, nama FROM kecamatan")
    df_old_kec = pd.DataFrame(cursor_old.fetchall())
    cursor_new.execute("SELECT id_kecamatan, id_kabupaten, nama_kecamatan FROM kecamatan")
    df_new_kec = pd.DataFrame(cursor_new.fetchall())

    cursor_old.execute("SELECT idkelurahan, idkecamatan, nama FROM kelurahan")
    df_old_kel = pd.DataFrame(cursor_old.fetchall())
    cursor_new.execute("SELECT id_kelurahan, id_kecamatan, nama_kelurahan FROM kelurahan")
    df_new_kel = pd.DataFrame(cursor_new.fetchall())

    def clean_wil_name(s):
        if pd.isna(s): return ""
        s = str(s).strip().lower()
        
        # Remove administrative words/abbreviations as whole words, including optional dot
        s = re.sub(r'\b(kabupaten|kab|kota|kecamatan|kec|kelurahan|kel|desa|adm)\b\.?', '', s)
        
        # Replace special punctuation/characters
        s = s.replace('\'', '').replace('`', '').replace('-', ' ')
        s = re.sub(r'\s+', ' ', s).strip()
        return s

    df_old_prov['clean'] = df_old_prov['nama'].apply(clean_wil_name)
    df_new_prov['clean'] = df_new_prov['nama_provinsi'].apply(clean_wil_name)
    df_old_kab['clean'] = df_old_kab['name'].apply(clean_wil_name)
    df_new_kab['clean'] = df_new_kab['nama_kabupaten'].apply(clean_wil_name)
    df_old_kec['clean'] = df_old_kec['nama'].apply(clean_wil_name)
    df_new_kec['clean'] = df_new_kec['nama_kecamatan'].apply(clean_wil_name)
    df_old_kel['clean'] = df_old_kel['nama'].apply(clean_wil_name)
    df_new_kel['clean'] = df_new_kel['nama_kelurahan'].apply(clean_wil_name)

    prov_map = {}
    for _, row in df_old_prov.iterrows():
        match = df_new_prov[df_new_prov['clean'] == row['clean']]
        if not match.empty:
            prov_map[row['idprovinsi']] = match.iloc[0]['id_provinsi']

    kab_map = {}
    df_new_kab['key'] = df_new_kab['clean'] + "_" + df_new_kab['id_provinsi'].astype(str)
    for _, row in df_old_kab.iterrows():
        new_prov_id = prov_map.get(row['idprovinsi'])
        if new_prov_id:
            key = row['clean'] + "_" + str(new_prov_id)
            match = df_new_kab[df_new_kab['key'] == key]
            if not match.empty:
                kab_map[row['idkabupaten']] = match.iloc[0]['id_kabupaten']

    kec_map = {}
    df_new_kec['key'] = df_new_kec['clean'] + "_" + df_new_kec['id_kabupaten'].astype(str)
    for _, row in df_old_kec.iterrows():
        new_kab_id = kab_map.get(row['idkabupaten'])
        if new_kab_id:
            key = row['clean'] + "_" + str(new_kab_id)
            match = df_new_kec[df_new_kec['key'] == key]
            if not match.empty:
                kec_map[row['idkecamatan']] = match.iloc[0]['id_kecamatan']

    kel_map = {}
    df_new_kel['key'] = df_new_kel['clean'] + "_" + df_new_kel['id_kecamatan'].astype(str)
    for _, row in df_old_kel.iterrows():
        new_kec_id = kec_map.get(row['idkecamatan'])
        if new_kec_id:
            key = row['clean'] + "_" + str(new_kec_id)
            match = df_new_kel[df_new_kel['key'] == key]
            if not match.empty:
                kel_map[row['idkelurahan']] = match.iloc[0]['id_kelurahan']

    df['id_provinsi'] = df['provinsi'].map(prov_map)
    df['id_kabupaten'] = df['kabupaten'].map(kab_map)
    df['id_kecamatan'] = df['kecamatan'].map(kec_map)
    df['id_kelurahan'] = df['kelurahan'].map(kel_map)
    
    # Normalisasi Agama
    agama_map = {
        'kristen': 'Kristen Protestan', 'protestan': 'Kristen Protestan', 
        'katholik': 'Katolik', 'budha': 'Buddha', 'khonghucu': 'Konghucu'
    }
    def normalize_agama(a):
        if pd.isna(a) or str(a).strip() == '': return 'Islam'
        a_clean = str(a).strip().lower()
        if 'kristen' in a_clean or 'protestan' in a_clean: return 'Kristen Protestan'
        if 'katholik' in a_clean or 'katolik' in a_clean: return 'Katolik'
        if 'hindu' in a_clean: return 'Hindu'
        if 'budha' in a_clean or 'buddha' in a_clean: return 'Buddha'
        if 'khonghucu' in a_clean or 'konghuchu' in a_clean: return 'Konghucu'
        return 'Islam'
    df['agama'] = df['agama'].apply(normalize_agama)

    mapping = {
        'idsiswa': 'id_siswa', 'tgl_daftar': 'tanggal_registrasi', 'domisili': 'domisili',
        'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin',
        'nama_sekolah': 'asal_sekolah', 'level_sekolah': 'tingkat_sekolah', 'nama_ortu': 'nama_orang_tua',
        'pekerjaan_ortu': 'pekerjaan_orang_tua', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir',
        'no_induk': 'nomor_induk', 'email': 'email', 'idcalon': 'id_calon',
        'id_provinsi': 'id_provinsi', 'id_kabupaten': 'id_kabupaten', 'id_kecamatan': 'id_kecamatan',
        'id_kelurahan': 'id_kelurahan', 'idmitra_int': 'id_mitra', 'nisn': 'nisn', 'nik': 'nik',
        'kewarganegaraan': 'kewarganegaraan', 'agama': 'agama', 'rt': 'rt', 'rw': 'rw',
        'kodepos': 'kode_pos', 'statussiswa': 'status_aktif', 'rekomen': 'rekomendasi',
        'info': 'sumber_info', 'pembayaran': 'metode_pembayaran', 'nama_ayah': 'nama_ayah',
        'pekerjaan_ayah': 'pekerjaan_ayah', 'jenjang_ayah': 'pendidikan_ayah', 
        'penghasilan_ayah': 'penghasilan_ayah', 'nama_ibu': 'nama_ibu', 'penghasilan_ibu': 'penghasilan_ibu',
        'jenjang_ibu': 'pendidikan_ibu', 'nama_wali': 'nama_wali', 'pekerjaan_wali': 'pekerjaan_wali',
        'jenjang_wali': 'pendidikan_wali', 'penghasilan_wali': 'penghasilan_wali',
        'wapeserta': 'wa_siswa', 'wawalmur': 'wa_ortu', 'waadmin': 'wa_administrasi',
        'sts_pengisian': 'status_pengisian', 'bukti': 'path_bukti_bayar', 'lulus': 'status_lulus_siswa',
        'created_bukti': 'tanggal_upload_bukti'
    }

    # Normalisasi Pekerjaan
    def normalize_pekerjaan(p):
        if pd.isna(p) or str(p).strip() in ('', '-', 'NO DATA', '0'): return 'Lainnya'
        s = str(p).strip().lower()
        if 'pegawai_swasta' in s or 'karyawan swasta' in s or 'karyawan' in s: return 'Pegawai Swasta'
        if 'wiraswasta' in s: return 'Wiraswasta'
        if 'aparatur_pejabat_negara' in s or 'tni' in s or 'pns' in s: return 'Aparatur/Pejabat Negara'
        if 'tenaga_kesehatan' in s: return 'Tenaga Kesehatan'
        if 'belum_tidak_bekerja' in s or 'tidak bekerja' in s: return 'Belum/Tidak Bekerja'
        if 'pensiunan' in s: return 'Pensiunan'
        if 'tenaga_pengajar' in s or 'guru' in s or 'dosen' in s: return 'Tenaga Pengajar'
        if 'agama_kepercayaan' in s: return 'Agama dan Kepercayaan'
        if 'pelajar_mahasiswa' in s or 'pelajar' in s: return 'Pelajar/Mahasiswa'
        if 'nelayan' in s: return 'Nelayan'
        if 'pertanian_peternakan' in s or 'tani' in s: return 'Pertanian/Peternakan'
        return 'Lainnya'
        
    df['pekerjaan_ayah'] = df['pekerjaan_ayah'].apply(normalize_pekerjaan)
    df['pekerjaan_wali'] = df['pekerjaan_wali'].apply(normalize_pekerjaan)
    if 'pekerjaan_ortu' in df.columns:
        df['pekerjaan_ortu'] = df['pekerjaan_ortu'].apply(normalize_pekerjaan)
        
    # Normalisasi Penghasilan
    def normalize_penghasilan(p):
        if pd.isna(p) or str(p).strip() in ('', '-', 'NO DATA', '0'): return None
        s = str(p).strip()
        if s in ('kurang_1jt', '1jt_3jt', '3jt_5jt', 'lebih_5jt'): return s
        return None
        
    df['penghasilan_ayah'] = df['penghasilan_ayah'].apply(normalize_penghasilan)
    df['penghasilan_ibu'] = df['penghasilan_ibu'].apply(normalize_penghasilan)
    df['penghasilan_wali'] = df['penghasilan_wali'].apply(normalize_penghasilan)
    
    df_final = df.rename(columns=mapping)
    df_final['pekerjaan_ibu'] = 'Lainnya'
    df_final['deleted_at'] = None
    target_cols = [c for c in list(mapping.values()) if c in df_final.columns] + ['pekerjaan_ibu', 'deleted_at']
    transformed_dfs['siswa'] = df_final[target_cols]

# 2. kursus_siswa
transformed_dfs['kursus_siswa'] = pd.DataFrame(columns=['id_kursus_siswa', 'id_siswa', 'id_kursus', 'tanggal_mulai', 'metode_belajar', 'status_aktif', 'catatan'])

# 3. siswa_keluar -> siswa_keluar
if 'siswa_keluar' in raw_data:
    df = pd.DataFrame(raw_data['siswa_keluar'])
    mapping = {
        'idsiswa_keluar': 'id_keluar', 'idsiswa': 'id_siswa',
        'alasan': 'alasan_keluar', 'tanggal': 'tanggal_keluar'
    }
    df = df.rename(columns=mapping)
    df['id_kursus'] = None
    
    # Heuristic for id_tag_keluar based on mapping.md "cek kolom alasan_keluar & keterangan_keluar"
    # Tag IDs from Fase 1 (1: Pindah, 2: Ekonomi, 3: Lulus, 4: Lainnya) - assuming heuristic
    def detect_tag(alasan):
        s = str(alasan).lower()
        if 'pindah' in s: return 1
        if 'ekonomi' in s or 'biaya' in s: return 2
        if 'lulus' in s or 'selesai' in s: return 3
        return 4 # Lainnya
        
    df['id_tag_keluar'] = df['alasan_keluar'].apply(detect_tag)
    transformed_dfs['siswa_keluar'] = df[list(mapping.values()) + ['id_kursus', 'id_tag_keluar']]

# 4. mitra -> mitra
if 'mitra' in raw_data:
    df = pd.DataFrame(raw_data['mitra'])
    df['id_mitra_new'] = df['idmitra'].apply(extract_int)
    df['kode_mitra'] = df['idmitra'].apply(extract_chars)
    
    df['provinsi_id'] = df['provinsi'].map(prov_map)
    df['kabupaten_id'] = df['kotkab'].map(kab_map)
    
    bool_cols = ['leapverse', 'kemitraan', 'elsa', 'classin', 'mitraleap']
    for col in bool_cols:
        df[col] = df[col].apply(convert_ya_tidak)
        
    mapping = {
        'id_mitra_new': 'id_mitra', 'nama': 'nama_mitra', 'instansi': 'nama_instansi',
        'namasekolah': 'nama_sekolah', 'lokasi': 'alamat_mitra', 'kepsek': 'nama_pimpinan',
        'cp': 'kontak_mitra', 'status': 'status_mitra', 'visimisi': 'visi_misi',
        'program': 'program_mitra', 'sdm': 'info_sdm', 'weakness': 'info_kelemahan',
        'rekomen': 'rekomendasi_program', 'jenis': 'jenis_mitra', 'provinsi_id': 'provinsi_id',
        'kabupaten_id': 'kabupaten_id', 'jml': 'jumlah_siswa_mitra', 'bidang': 'bidang_usaha',
        'leapverse': 'is_leapverse', 'kemitraan': 'status_kemitraan', 'tahun': 'tahun_bergabung',
        'jeniskemitraan': 'tipe_kerjasama', 'elsa': 'is_elsa', 'classin': 'is_classin',
        'mitraleap': 'is_mitra_leap', 'created_at': 'created_at', 'kode_mitra': 'kode_mitra'
    }
    transformed_dfs['mitra'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 5. mitra_note -> mitra_progres
if 'mitra_note' in raw_data:
    df = pd.DataFrame(raw_data['mitra_note'])
    mapping = {
        'idmnote': 'id_progres_mitra', 'idmitra': 'id_mitra',
        'note': 'catatan_progres_mitra', 'idusers': 'id_user', 'status': 'status_progres_mitra',
        'startdate': 'kemitraan_mulai', 'enddate': 'kemitraan_berakhir', 'created_at': 'created_at'
    }
    transformed_dfs['mitra_progres'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 6. mitra_users -> kemitraan_verifikator
if 'mitra_users' in raw_data:
    df = pd.DataFrame(raw_data['mitra_users'])
    mapping = {
        'idmusers': 'id_kemitraan', 'idmnote': 'id_progres_mitra', 'idusers': 'id_user'
    }
    transformed_dfs['kemitraan_verifikator'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

# 7. siswamitra -> siswa_mitra
if 'siswamitra' in raw_data:
    df = pd.DataFrame(raw_data['siswamitra'])
    mapping = {
        'idsiswa': 'id_sm', 'tgl_daftar': 'tanggal_daftar', 'domisili': 'alamat_domisili',
        'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin',
        'nama_instansi': 'nama_instansi', 'level_sekolah': 'tingkat_sekolah',
        'pekerjaan': 'pekerjaan_sm', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir',
        'no_induk': 'nomor_induk_sm', 'email': 'email_sm', 'tlp': 'wa_sm',
        'keluar': 'status_keluar_sm', 'idmitra': 'id_mitra'
    }
    df_final = df.rename(columns=mapping)
    df_final['sertifikat_sm'] = None
    transformed_dfs['siswa_mitra'] = df_final.reindex(columns=list(mapping.values()) + ['sertifikat_sm'])

# 8. siswa_keluar_mitra -> siswa_mitra_keluar
if 'siswa_keluar_mitra' in raw_data:
    df = pd.DataFrame(raw_data['siswa_keluar_mitra'])
    mapping = {
        'idsiswa_keluar': 'id_sm_keluar', 'idsiswa': 'id_sm',
        'alasan': 'alasan_keluar_sm', 'tanggal': 'tanggal_keluar_sm'
    }
    transformed_dfs['siswa_mitra_keluar'] = df.rename(columns=mapping).reindex(columns=list(mapping.values()))

print(f"✓ Transformasi {len(transformed_dfs)} tabel Fase 4 selesai.")


✓ Transformasi 8 tabel Fase 4 selesai.


## 3.1 Verifikasi Hasil Transformasi
Bagian ini menampilkan perbandingan jumlah data dan tipe data untuk pengecekan manual.

In [5]:
# 3.1.1 Ringkasan Jumlah Baris
print("📊 RINGKASAN MIGRASI (RECORDS COUNT)")
print("="*70)
summary_list = []
for old_t, new_t in hanif_tables_map:
    old_c = len(raw_data.get(old_t, []))
    new_c = len(transformed_dfs.get(new_t, []))
    summary_list.append({
        'Tabel Lama': old_t,
        'Tabel Baru': new_t,
        'Old Recs': old_c,
        'New Recs': new_c,
        'Diff': new_c - old_c,
        'Status': "✅ OK" if old_c == new_c else "⚠️ Cek"
    })
display(pd.DataFrame(summary_list))

total_old_all = sum(len(records) for records in raw_data.values())
total_new_all = sum(len(df) for df in transformed_dfs.values())
print(f"\n📢 TOTAL REKAPITULASI: {total_old_all} (Old) ➔ {total_new_all} (New)")
if total_old_all == total_new_all: print("✅ SEMUA DATA TERANGKUT")
else: print(f"⚠️ ADA SELISIH: {total_new_all - total_old_all} baris")

📊 RINGKASAN MIGRASI (RECORDS COUNT)


,Tabel Lama,Tabel Baru,Old Recs,New Recs,Diff,Status
0,siswa,siswa,1469,1469,0,✅ OK
1,siswa_keluar,siswa_keluar,556,556,0,✅ OK
2,mitra,mitra,22,22,0,✅ OK
3,mitra_note,mitra_progres,296,296,0,✅ OK
4,mitra_users,kemitraan_verifikator,228,228,0,✅ OK
5,siswamitra,siswa_mitra,0,0,0,✅ OK
6,siswa_keluar_mitra,siswa_mitra_keluar,0,0,0,✅ OK



📢 TOTAL REKAPITULASI: 2571 (Old) ➔ 2571 (New)
✅ SEMUA DATA TERANGKUT


In [6]:
# 3.1.2 Output Pengecekan Kolom Spesifik (KETERANGAN mapping.md)
print("\n🔍 PENGECEKAN TRANSFORMASI SPESIFIK (KETERANGAN mapping.md)")
print("="*70)

# 1. Pengecekan ID Mitra (Siswa)
if 'siswa' in transformed_dfs:
    print("\n[SISWA] Pengecekan Ekstraksi ID Mitra (Int):")
    display(transformed_dfs['siswa'][['nama_lengkap', 'id_mitra']].dropna(subset=['id_mitra']).head(5))

# 2. Pengecekan Agama (Siswa)
if 'siswa' in transformed_dfs:
    print("\n[SISWA] Pengecekan Normalisasi Agama:")
    display(transformed_dfs['siswa']['agama'].value_counts())

# 3. Pengecekan Mitra (Boolean & ID/Kode)
if 'mitra' in transformed_dfs:
    print("\n[MITRA] Pengecekan Boolean (Ya/Tidak -> 1/0) & Kode Mitra:")
    display(transformed_dfs['mitra'][['nama_mitra', 'id_mitra', 'kode_mitra', 'is_leapverse', 'status_kemitraan']].head(5))

# 4. Pengecekan Audit Trailing (Mitra & Progres)
if 'mitra' in transformed_dfs:
    print("\n[MITRA] Pengecekan created_at (Direct Mapping):")
    display(transformed_dfs['mitra'][['nama_mitra', 'created_at']].head(5))


🔍 PENGECEKAN TRANSFORMASI SPESIFIK (KETERANGAN mapping.md)

[SISWA] Pengecekan Ekstraksi ID Mitra (Int):


,nama_lengkap,id_mitra
136,FAUZIAH MARTATI,25.0
334,SHERLY INDRIA BUDIANDA,25.0
433,FITA EMIRIZA,13.0
434,SITI NUR MAGHFIROH,13.0
435,DWI YUNIAR RIANAWATI,13.0



[SISWA] Pengecekan Normalisasi Agama:


agama
Islam                1218
Kristen Protestan     162
Katolik                71
Hindu                  15
Konghucu                3
Name: count, dtype: int64


[MITRA] Pengecekan Boolean (Ya/Tidak -> 1/0) & Kode Mitra:


,nama_mitra,id_mitra,kode_mitra,is_leapverse,status_kemitraan
0,Fiona Febianita Sulistyo,2,M,0,0
1,Chelsea,3,M,0,0
2,Geraldo P. Latumahina,6,M,0,0
3,Anggi dewantoro,7,M,0,0
4,Susanti,8,M,0,0



[MITRA] Pengecekan created_at (Direct Mapping):


,nama_mitra,created_at
0,Fiona Febianita Sulistyo,2023-09-04 07:06:34
1,Chelsea,2023-10-23 08:28:10
2,Geraldo P. Latumahina,2023-12-04 04:56:14
3,Anggi dewantoro,2024-08-27 07:02:29
4,Susanti,2024-08-27 07:20:16


In [7]:
# 3.1.3 Detail Perbandingan Kolom & Tipe Data (Side-by-Side)
print("\n🔍 PERBANDINGAN TIPE DATA SIDE-BY-SIDE")
for old_t, new_t in hanif_tables_map:
    print(f"\n{'='*15} {old_t.upper()} ➔ {new_t.upper()} {'='*15}")
    
    df_old = pd.DataFrame(raw_data.get(old_t, []))
    df_new = transformed_dfs.get(new_t, pd.DataFrame())
    
    if not df_new.empty or not df_old.empty:
        comparison = []
        table_mapping = {}
        if old_t == 'siswa': table_mapping = {'idsiswa': 'id_siswa', 'tgl_daftar': 'tanggal_registrasi', 'domisili': 'domisili', 'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin', 'nama_sekolah': 'asal_sekolah', 'level_sekolah': 'tingkat_sekolah', 'nama_ortu': 'nama_orang_tua', 'pekerjaan_ortu': 'pekerjaan_orang_tua', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir', 'no_induk': 'nomor_induk', 'email': 'email', 'idcalon': 'id_calon', 'provinsi': 'id_provinsi', 'kabupaten': 'id_kabupaten', 'kecamatan': 'id_kecamatan', 'kelurahan': 'id_kelurahan', 'idmitra': 'id_mitra', 'nisn': 'nisn', 'nik': 'nik', 'kewarganegaraan': 'kewarganegaraan', 'agama': 'agama', 'rt': 'rt', 'rw': 'rw', 'kodepos': 'kode_pos', 'statussiswa': 'status_aktif', 'rekomen': 'rekomendasi', 'info': 'sumber_info', 'pembayaran': 'metode_pembayaran', 'nama_ayah': 'nama_ayah', 'pekerjaan_ayah': 'pekerjaan_ayah', 'jenjang_ayah': 'pendidikan_ayah', 'penghasilan_ayah': 'penghasilan_ayah', 'nama_ibu': 'nama_ibu', 'penghasilan_ibu': 'penghasilan_ibu', 'jenjang_ibu': 'pendidikan_ibu', 'nama_wali': 'nama_wali', 'pekerjaan_wali': 'pekerjaan_wali', 'jenjang_wali': 'pendidikan_wali', 'penghasilan_wali': 'penghasilan_wali', 'wapeserta': 'wa_siswa', 'wawalmur': 'wa_ortu', 'waadmin': 'wa_administrasi', 'sts_pengisian': 'status_pengisian', 'bukti': 'path_bukti_bayar', 'lulus': 'status_lulus_siswa', 'created_at': 'created_at'}
        elif old_t == 'siswa_keluar': table_mapping = {'idsiswa_keluar': 'id_keluar', 'idsiswa': 'id_siswa', 'alasan': 'alasan_keluar', 'tanggal': 'tanggal_keluar'}
        elif old_t == 'mitra': table_mapping = {'idmitra': 'id_mitra', 'nama': 'nama_mitra', 'instansi': 'nama_instansi', 'namasekolah': 'nama_sekolah', 'lokasi': 'alamat_mitra', 'kepsek': 'nama_pimpinan', 'cp': 'kontak_mitra', 'status': 'status_mitra', 'visimisi': 'visi_misi', 'program': 'program_mitra', 'sdm': 'info_sdm', 'weakness': 'info_kelemahan', 'rekomen': 'rekomendasi_program', 'jenis': 'jenis_mitra', 'provinsi': 'provinsi_id', 'kotkab': 'kabupaten_id', 'jml': 'jumlah_siswa_mitra', 'bidang': 'bidang_usaha', 'leapverse': 'is_leapverse', 'kemitraan': 'status_kemitraan', 'tahun': 'tahun_bergabung', 'jeniskemitraan': 'tipe_kerjasama', 'elsa': 'is_elsa', 'classin': 'is_classin', 'mitraleap': 'is_mitra_leap', 'created_at': 'created_at'}
        elif old_t == 'mitra_note': table_mapping = {'idmnote': 'id_progres_mitra', 'idmitra': 'id_mitra', 'note': 'catatan_progres_mitra', 'idusers': 'id_user', 'status': 'status_progres_mitra', 'startdate': 'kemitraan_mulai', 'enddate': 'kemitraan_berakhir', 'created_at': 'created_at'}
        elif old_t == 'mitra_users': table_mapping = {'idmusers': 'id_kemitraan', 'idmnote': 'id_progres_mitra', 'idusers': 'id_user'}
        elif old_t == 'siswamitra': table_mapping = {'idsiswa': 'id_sm', 'tgl_daftar': 'tanggal_daftar', 'domisili': 'alamat_domisili', 'nama_lengkap': 'nama_lengkap', 'panggilan': 'nama_panggilan', 'jkel': 'jenis_kelamin', 'nama_instansi': 'nama_instansi', 'level_sekolah': 'tingkat_sekolah', 'pekerjaan': 'pekerjaan_sm', 'tmp_lahir': 'tempat_lahir', 'tgl_lahir': 'tanggal_lahir', 'no_induk': 'nomor_induk_sm', 'email': 'email_sm', 'tlp': 'wa_sm', 'keluar': 'status_keluar_sm', 'idmitra': 'id_mitra'}
        elif old_t == 'siswa_keluar_mitra': table_mapping = {'idsiswa_keluar': 'id_sm_keluar', 'idsiswa': 'id_sm', 'alasan': 'alasan_keluar_sm', 'tanggal': 'tanggal_keluar_sm'}

        for old_col, new_col in table_mapping.items():
            comparison.append({
                'Old Column': old_col,
                'Old Type': str(df_old[old_col].dtype) if not df_old.empty and old_col in df_old.columns else "N/A",
                '➔': '➔',
                'New Column': new_col,
                'New Type': str(df_new[new_col].dtype) if not df_new.empty and new_col in df_new.columns else "N/A"
            })
        
        # Cek kolom baru
        if not df_new.empty:
            for col in df_new.columns:
                if col not in table_mapping.values():
                    comparison.append({
                        'Old Column': '(KOLOM BARU / CUSTOM)',
                        'Old Type': '-',
                        '➔': '➔',
                        'New Column': col,
                        'New Type': str(df_new[col].dtype)
                    })
        
        display(pd.DataFrame(comparison))
        if not df_new.empty:
            print(f"\n--- SAMPLE DATA NEW (2 Baris) ---")
            display(df_new.head(2))
    else:
        print(f"⚠️ Tabel {new_t} kosong.")


🔍 PERBANDINGAN TIPE DATA SIDE-BY-SIDE

=============== SISWA ➔ SISWA ===============


,Old Column,Old Type,➔,New Column,New Type
0,idsiswa,object,➔,id_siswa,object
1,tgl_daftar,object,➔,tanggal_registrasi,object
2,domisili,object,➔,domisili,object
3,nama_lengkap,object,➔,nama_lengkap,object
4,panggilan,object,➔,nama_panggilan,object
5,jkel,object,➔,jenis_kelamin,object
6,nama_sekolah,object,➔,asal_sekolah,object
7,level_sekolah,object,➔,tingkat_sekolah,object
8,nama_ortu,object,➔,nama_orang_tua,object
9,pekerjaan_ortu,object,➔,pekerjaan_orang_tua,object



--- SAMPLE DATA NEW (2 Baris) ---


,id_siswa,tanggal_registrasi,domisili,nama_lengkap,nama_panggilan,jenis_kelamin,asal_sekolah,tingkat_sekolah,nama_orang_tua,pekerjaan_orang_tua,...,penghasilan_wali,wa_siswa,wa_ortu,wa_administrasi,status_pengisian,path_bukti_bayar,status_lulus_siswa,tanggal_upload_bukti,pekerjaan_ibu,deleted_at
0,S0000007,2022-07-01,Rungkut Barata VI/12-14,EZRA RAFA DANAR,RAFA,Laki-laki,MIN 1 Medokan Ayu,SD,IBU EZRA RAFA DANAR (Nur Arief),Belum/Tidak Bekerja,...,kurang_1jt,,085230012257,085230012257,Sudah Lengkap,None,0.0,None,Lainnya,None
1,S0000008,None,,SARAH MEDINA ISWALDI,SARAH,Laki-laki,,,IBU SARAH,Lainnya,...,None,None,None,None,Belum Lengkap,None,0.0,None,Lainnya,None



=============== SISWA_KELUAR ➔ SISWA_KELUAR ===============


...


=============== MITRA_USERS ➔ KEMITRAAN_VERIFIKATOR ===============


,Old Column,Old Type,➔,New Column,New Type
0,idmusers,object,➔,id_kemitraan,object
1,idmnote,object,➔,id_progres_mitra,object
2,idusers,object,➔,id_user,object



--- SAMPLE DATA NEW (2 Baris) ---


,id_kemitraan,id_progres_mitra,id_user
0,P00005,N00007,U00011
1,P00006,N00008,U00011



=============== SISWAMITRA ➔ SISWA_MITRA ===============
⚠️ Tabel siswa_mitra kosong.

=============== SISWA_KELUAR_MITRA ➔ SISWA_MITRA_KELUAR ===============
⚠️ Tabel siswa_mitra_keluar kosong.


## 4. Export ke Pickle

In [8]:
file_name = 'fase_4_hanif.pkl'

# Fix: Konversi tipe data StringDtype ke object agar kompatibel dengan Python 3.13 pickle
for table in transformed_dfs:
    df = transformed_dfs[table]
    if not df.empty:
        for col in df.columns:
            if str(df[col].dtype) in ['string', 'string[python]']:
                df[col] = df[col].astype(object)

pd.to_pickle(transformed_dfs, file_name)

total_records_new = sum(len(df) for df in transformed_dfs.values())
total_records_old = sum(len(records) for records in raw_data.values())

migration_result = {
    'fase': 'fase_4',
    'script': 'script_hanif',
    'fase_num': 4,
    'status': 'ready_for_insert',
    'old_records_total': total_records_old,
    'new_records_total': total_records_new,
    'diff': total_records_new - total_records_old,
    'pickle_file': file_name,
    'timestamp': datetime.now().isoformat()
}
print(json.dumps(migration_result, indent=2))

cursor_old.close()
cursor_new.close()
db_old.close()
db_new.close()

{
  "fase": "fase_4",
  "script": "script_hanif",
  "fase_num": 4,
  "status": "ready_for_insert",
  "old_records_total": 2571,
  "new_records_total": 2571,
  "diff": 0,
  "pickle_file": "fase_4_hanif.pkl",
  "timestamp": "2026-05-25T14:19:07.537605"
}
